In [1]:
import pandas as pd
df = pd.read_csv("spotify_tracks.csv")

In [6]:
# Keeps rows where the column value matches anything in the list
df = df[df['language'].isin(['English', 'Hindi'])]
df.shape


(29132, 22)

In [7]:
df.to_pickle("song_catalog.pkl")

In [8]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ============================================================
# CONFIGURATION
# ============================================================

N_USERS = 1000
MIN_INTERACTIONS_PER_USER = 80
MAX_INTERACTIONS_PER_USER = 100

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ============================================================
# LOAD SONG CATALOG
# ============================================================

df = pd.read_pickle("song_catalog.pkl").copy()

feature_columns = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "tempo",
    "valence"
]

# Keep only songs with complete feature information
df = df.dropna(
    subset=feature_columns
).reset_index(drop=True)

print("Song catalog:", df.shape)

# ============================================================
# CREATE NORMALIZED FEATURES
# ============================================================

# Convert every feature to approximately 0-1 range
# This is only for synthetic-data generation.
# It does NOT replace the scaler used by V1.

song_features = df[feature_columns].copy()

for col in feature_columns:

    min_value = song_features[col].min()
    max_value = song_features[col].max()

    if max_value == min_value:
        song_features[col] = 0.5
    else:
        song_features[col] = (
            song_features[col] - min_value
        ) / (
            max_value - min_value
        )

# Tempo is special because its raw range is much wider.
# After normalization above it is already approximately 0-1.

song_matrix = song_features.values

# ============================================================
# USER ARCHETYPES
# ============================================================

archetypes = {
    
    "high_energy_workout": {
        "acousticness": 0.15,
        "danceability": 0.80,
        "energy": 0.92,
        "instrumentalness": 0.10,
        "liveness": 0.25,
        "loudness": 0.85,
        "speechiness": 0.15,
        "tempo": 0.80,
        "valence": 0.80
    },

    "edm_fan": {
        "acousticness": 0.08,
        "danceability": 0.88,
        "energy": 0.94,
        "instrumentalness": 0.45,
        "liveness": 0.20,
        "loudness": 0.90,
        "speechiness": 0.08,
        "tempo": 0.82,
        "valence": 0.75
    },

    "pop_listener": {
        "acousticness": 0.20,
        "danceability": 0.78,
        "energy": 0.72,
        "instrumentalness": 0.05,
        "liveness": 0.15,
        "loudness": 0.70,
        "speechiness": 0.10,
        "tempo": 0.60,
        "valence": 0.78
    },

    "rock_fan": {
        "acousticness": 0.18,
        "danceability": 0.50,
        "energy": 0.90,
        "instrumentalness": 0.20,
        "liveness": 0.35,
        "loudness": 0.95,
        "speechiness": 0.08,
        "tempo": 0.70,
        "valence": 0.60
    },

    "chill_listener": {
        "acousticness": 0.75,
        "danceability": 0.45,
        "energy": 0.30,
        "instrumentalness": 0.25,
        "liveness": 0.10,
        "loudness": 0.35,
        "speechiness": 0.05,
        "tempo": 0.35,
        "valence": 0.55
    },

    "acoustic_listener": {
        "acousticness": 0.90,
        "danceability": 0.40,
        "energy": 0.35,
        "instrumentalness": 0.15,
        "liveness": 0.20,
        "loudness": 0.30,
        "speechiness": 0.08,
        "tempo": 0.40,
        "valence": 0.60
    },

    "hiphop_listener": {
        "acousticness": 0.20,
        "danceability": 0.78,
        "energy": 0.75,
        "instrumentalness": 0.10,
        "liveness": 0.20,
        "loudness": 0.75,
        "speechiness": 0.55,
        "tempo": 0.55,
        "valence": 0.65
    },

    "low_energy_listener": {
        "acousticness": 0.60,
        "danceability": 0.40,
        "energy": 0.25,
        "instrumentalness": 0.20,
        "liveness": 0.15,
        "loudness": 0.30,
        "speechiness": 0.10,
        "tempo": 0.30,
        "valence": 0.40
    }
}

archetype_names = list(archetypes.keys())

# ============================================================
# GENERATE SYNTHETIC USERS
# ============================================================

users = []
profiles = []

for i in range(N_USERS):

    user_id = f"U{i+1:04d}"

    # Randomly select an archetype
    archetype = np.random.choice(archetype_names)

    base_profile = archetypes[archetype]

    # Add individual variation
    profile = {}

    for feature in feature_columns:

        base_value = base_profile[feature]

        # Small individual variation
        variation = np.random.normal(
            loc=0,
            scale=0.08
        )

        value = base_value + variation

        # Keep values between 0 and 1
        value = np.clip(value, 0, 1)

        profile[feature] = value

    users.append({
        "user_id": user_id,
        "archetype": archetype
    })

    profile_row = {
        "user_id": user_id,
        "archetype": archetype
    }

    profile_row.update(profile)

    profiles.append(profile_row)

synthetic_users = pd.DataFrame(users)
user_profiles = pd.DataFrame(profiles)

print("Synthetic users:", synthetic_users.shape)
print("User profiles:", user_profiles.shape)

# ============================================================
# GENERATE INTERACTIONS
# ============================================================

interactions = []

# Convert song duration to seconds where available
if "duration_ms" in df.columns:
    song_duration = (
        df["duration_ms"]
        .fillna(df["duration_ms"].median())
        / 1000
    ).values
else:
    song_duration = np.full(
        len(df),
        200
    )

# Current date used as the endpoint for synthetic history
end_date = datetime.now()

for user_index in range(N_USERS):

    user_id = f"U{user_index+1:04d}"

    user_profile = user_profiles.iloc[user_index]

    user_vector = np.array([
        user_profile[feature]
        for feature in feature_columns
    ])

    # --------------------------------------------------------
    # Calculate similarity between user and every song
    # --------------------------------------------------------

    distances = np.mean(
        np.abs(
            song_matrix - user_vector
        ),
        axis=1
    )

    # Convert distance into preference score
    preference_scores = np.exp(
        -4 * distances
    )

    # --------------------------------------------------------
    # Add exploration
    # --------------------------------------------------------

    # 85% preference-driven
    # 15% exploration/randomness

    preference_probabilities = (
        0.85 * preference_scores
        + 0.15 * np.random.random(
            len(df)
        )
    )

    # Prevent numerical problems
    preference_probabilities = np.maximum(
        preference_probabilities,
        1e-10
    )

    # Normalize into probability distribution
    preference_probabilities /= (
        preference_probabilities.sum()
    )

    # --------------------------------------------------------
    # Select songs the user will interact with
    # --------------------------------------------------------
# Each user has a random number of interactions between 80 and 100
    interactions_per_user = np.random.randint(
        MIN_INTERACTIONS_PER_USER,
        MAX_INTERACTIONS_PER_USER + 1
    )

    selected_indices = np.random.choice(
        len(df),
        size=interactions_per_user,
        replace=False,
        p=preference_probabilities
    )

    for song_index in selected_indices:

        song_score = preference_scores[song_index]

        # Add noise so behavior isn't perfectly deterministic
        behavioral_score = (
            song_score
            + np.random.normal(
                0,
                0.08
            )
        )

        behavioral_score = np.clip(
            behavioral_score,
            0,
            1
        )

        # ----------------------------------------------------
        # Determine interaction type
        # ----------------------------------------------------

        random_value = np.random.random()

        if behavioral_score >= 0.78:

            if random_value < 0.15:
                event = "like"

            elif random_value < 0.85:
                event = "complete"

            else:
                event = "play"

        elif behavioral_score >= 0.58:

            if random_value < 0.65:
                event = "complete"

            elif random_value < 0.90:
                event = "play"

            else:
                event = "skip"

        elif behavioral_score >= 0.40:

            if random_value < 0.45:
                event = "play"

            elif random_value < 0.85:
                event = "skip"

            else:
                event = "complete"

        else:

            if random_value < 0.70:
                event = "skip"

            elif random_value < 0.90:
                event = "dislike"

            else:
                event = "play"

        # ----------------------------------------------------
        # Completion rate
        # ----------------------------------------------------

        if event == "complete":

            completion_rate = np.random.uniform(
                0.75,
                1.00
            )

        elif event == "like":

            completion_rate = np.random.uniform(
                0.85,
                1.00
            )

        elif event == "play":

            completion_rate = np.random.uniform(
                0.25,
                0.75
            )

        elif event == "skip":

            completion_rate = np.random.uniform(
                0.00,
                0.25
            )

        else:
            completion_rate = np.random.uniform(
                0.00,
                0.15
            )

        # ----------------------------------------------------
        # Play duration
        # ----------------------------------------------------

        duration = song_duration[song_index]

        play_duration = (
            duration * completion_rate
        )

        # ----------------------------------------------------
        # Rating
        # ----------------------------------------------------

        if event == "like":
            rating = 5

        elif event == "complete":
            rating = np.random.choice(
                [3, 4, 5],
                p=[0.10, 0.45, 0.45]
            )

        elif event == "play":
            rating = np.random.choice(
                [2, 3, 4],
                p=[0.20, 0.50, 0.30]
            )

        elif event == "skip":
            rating = np.random.choice(
                [1, 2],
                p=[0.70, 0.30]
            )

        else:
            rating = 1

        # ----------------------------------------------------
        # Timestamp
        # ----------------------------------------------------

        random_days = np.random.randint(
            0,
            180
        )

        random_seconds = np.random.randint(
            0,
            86400
        )

        timestamp = (
            end_date
            - timedelta(
                days=int(random_days),
                seconds=int(random_seconds)
            )
        )

        # ----------------------------------------------------
        # Store interaction
        # ----------------------------------------------------

        track_id = df.iloc[
            song_index
        ]["track_id"]

        interactions.append({

            "user_id": user_id,

            "track_id": track_id,

            "timestamp": timestamp,

            "event": event,

            "play_duration_seconds": round(
                play_duration,
                2
            ),

            "completion_rate": round(
                completion_rate,
                3
            ),

            "rating": rating,

            "preference_score": round(
                behavioral_score,
                4
            )
        })

# ============================================================
# CREATE DATAFRAME
# ============================================================

synthetic_interactions = pd.DataFrame(
    interactions
)

synthetic_interactions = (
    synthetic_interactions
    .sort_values(
        ["user_id", "timestamp"]
    )
    .reset_index(drop=True)
)

# ============================================================
# CREATE TRAIN / VALIDATION / TEST SPLIT
# ============================================================

synthetic_interactions["split"] = ""

for user_id, group in synthetic_interactions.groupby(
    "user_id"
):

    indices = group.index.to_numpy().copy()

    np.random.shuffle(indices)

    n = len(indices)

    train_end = int(
        n * 0.70
    )

    validation_end = int(
        n * 0.85
    )

    train_indices = indices[
        :train_end
    ]

    validation_indices = indices[
        train_end:validation_end
    ]

    test_indices = indices[
        validation_end:
    ]

    synthetic_interactions.loc[
        train_indices,
        "split"
    ] = "train"

    synthetic_interactions.loc[
        validation_indices,
        "split"
    ] = "validation"

    synthetic_interactions.loc[
        test_indices,
        "split"
    ] = "test"

# ============================================================
# SAVE DATASETS
# ============================================================

synthetic_users.to_csv(
    "synthetic_users.csv",
    index=False
)

user_profiles.to_csv(
    "user_profiles.csv",
    index=False
)

synthetic_interactions.to_csv(
    "synthetic_interactions.csv",
    index=False
)

# Separate split files
synthetic_interactions[
    synthetic_interactions["split"] == "train"
].to_csv(
    "interactions_train.csv",
    index=False
)

synthetic_interactions[
    synthetic_interactions["split"] == "validation"
].to_csv(
    "interactions_validation.csv",
    index=False
)

synthetic_interactions[
    synthetic_interactions["split"] == "test"
].to_csv(
    "interactions_test.csv",
    index=False
)

# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("V2 SYNTHETIC DATASET CREATED")
print("=" * 60)

print(
    f"\nUsers: {len(synthetic_users):,}"
)

print(
    f"Total interactions: "
    f"{len(synthetic_interactions):,}"
)

print(
    f"Unique songs interacted with: "
    f"{synthetic_interactions['track_id'].nunique():,}"
)

print(
    f"\nTrain interactions: "
    f"{len(synthetic_interactions[synthetic_interactions['split'] == 'train']):,}"
)

print(
    f"Validation interactions: "
    f"{len(synthetic_interactions[synthetic_interactions['split'] == 'validation']):,}"
)

print(
    f"Test interactions: "
    f"{len(synthetic_interactions[synthetic_interactions['split'] == 'test']):,}"
)

print("\nInteraction distribution:")
print(
    synthetic_interactions["event"]
    .value_counts()
)

print("\nUser archetype distribution:")
print(
    synthetic_users["archetype"]
    .value_counts()
)

print("\nFiles created:")
print("  synthetic_users.csv")
print("  user_profiles.csv")
print("  synthetic_interactions.csv")
print("  interactions_train.csv")
print("  interactions_validation.csv")
print("  interactions_test.csv")

print("\nSample interactions:")
display(
    synthetic_interactions.head(10)
)

print("\nSample user profiles:")
display(
    user_profiles.head(10)
)

Song catalog: (29132, 22)
Synthetic users: (1000, 2)
User profiles: (1000, 11)

V2 SYNTHETIC DATASET CREATED

Users: 1,000
Total interactions: 90,258
Unique songs interacted with: 27,816

Train interactions: 62,695
Validation interactions: 13,571
Test interactions: 13,992

Interaction distribution:
event
skip        57632
play        15315
dislike     14214
complete     3096
like            1
Name: count, dtype: int64

User archetype distribution:
archetype
high_energy_workout    134
hiphop_listener        133
edm_fan                132
acoustic_listener      131
chill_listener         127
rock_fan               121
low_energy_listener    115
pop_listener           107
Name: count, dtype: int64

Files created:
  synthetic_users.csv
  user_profiles.csv
  synthetic_interactions.csv
  interactions_train.csv
  interactions_validation.csv
  interactions_test.csv

Sample interactions:


,user_id,track_id,timestamp,event,play_duration_seconds,completion_rate,rating,preference_score,split
0,U0001,4XkajNf3Z2zDW5WcTjmPLh,2026-03-07 11:33:54.516355,complete,212.28,0.946,4,0.5981,train
1,U0001,2caIW6c3miNpJM40xx330C,2026-03-07 22:12:56.516355,skip,50.01,0.197,1,0.4173,validation
2,U0001,3sv4HSEOAAnGdGs76UFnfC,2026-03-08 17:53:24.516355,skip,71.32,0.250,1,0.4305,train
3,U0001,7crZfJQOb8QsLtn3LwwQLK,2026-03-09 06:16:30.516355,play,136.17,0.492,3,0.6128,train
4,U0001,2dfH1gA3FZDv9L69Dqy9RY,2026-03-09 07:59:47.516355,dislike,19.08,0.118,1,0.3227,train
5,U0001,59Fu6rrNI4ecUBqKxRMz4x,2026-03-15 09:59:27.516355,dislike,2.87,0.010,1,0.3652,train
6,U0001,3Af5643muHHTzAkkGijser,2026-03-15 22:35:18.516355,play,74.62,0.389,2,0.4697,validation
7,U0001,7JeWimOydExpA4mwyknLLz,2026-03-17 13:12:45.516355,skip,12.69,0.045,1,0.3153,train
8,U0001,6wGx1gGCA9jBz5MCGn0FeU,2026-03-18 01:33:25.516355,skip,66.79,0.244,2,0.2605,train
9,U0001,3DzA9hlVPUVZ2UucmsQQxY,2026-03-19 09:08:53.516355,dislike,4.14,0.021,1,0.3353,train



Sample user profiles:


,user_id,archetype,acousticness,danceability,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence
0,U0001,hiphop_listener,0.155981,0.821235,0.787909,0.209476,0.126654,0.740068,0.389123,0.510576,0.681406
1,U0002,chill_listener,0.675665,0.403530,0.257986,0.204290,0.026073,0.140996,0.126030,0.415316,0.428090
2,U0003,edm_fan,0.059916,0.866891,0.821894,0.568958,0.198044,0.928444,0.113361,0.886597,0.726528
3,U0004,high_energy_workout,0.147613,0.751949,0.896665,0.051863,0.398182,0.848920,0.065383,0.865804,0.702333
4,U0005,edm_fan,0.000000,0.851751,0.903083,0.455333,0.185897,0.996071,0.135872,0.806270,0.677425
5,U0006,acoustic_listener,0.995090,0.319273,0.223336,0.211896,0.156949,0.192266,0.009553,0.309556,0.610754
6,U0007,hiphop_listener,0.049946,0.670657,0.800904,0.027462,0.238083,0.854293,0.566927,0.597764,0.578293
7,U0008,hiphop_listener,0.191041,0.691493,0.654303,0.165002,0.308499,0.744239,0.630283,0.578931,0.598390
8,U0009,high_energy_workout,0.075299,0.862737,0.870810,0.126631,0.139543,0.827535,0.145215,0.876894,0.943542
9,U0010,chill_listener,0.796455,0.407247,0.180396,0.186859,0.159497,0.333027,0.015850,0.390116,0.642655


In [10]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_pickle("song_catalog.pkl").copy()

train = pd.read_csv("interactions_train.csv")
validation = pd.read_csv("interactions_validation.csv")
test = pd.read_csv("interactions_test.csv")


# ============================================================
# 2. FEATURES
# ============================================================

feature_columns = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "tempo",
    "valence"
]


# ============================================================
# 3. CLEAN DATA
# ============================================================

df = df.dropna(
    subset=feature_columns
).reset_index(drop=True)

df = (
    df
    .drop_duplicates(subset=["track_id"])
    .dropna(subset=feature_columns)
    .reset_index(drop=True)
)


train = train[
    train["track_id"].isin(df["track_id"])
].copy()

validation = validation[
    validation["track_id"].isin(df["track_id"])
].copy()

test = test[
    test["track_id"].isin(df["track_id"])
].copy()


# ============================================================
# 4. CREATE SONG FEATURE MATRIX
# ============================================================

song_features = df[feature_columns].copy()

scaler = StandardScaler()

song_features_scaled = scaler.fit_transform(
    song_features
)

song_feature_matrix = pd.DataFrame(
    song_features_scaled,
    index=df["track_id"],
    columns=feature_columns
)

print("Total songs:", len(df))
print("Unique track IDs:", df["track_id"].nunique())
print("Duplicate track IDs:", df["track_id"].duplicated().sum())


# ============================================================
# 5. INTERACTION WEIGHTS
# ============================================================

interaction_weights = {
    "like": 5.0,
    "complete": 3.0,
    "play": 1.0,
    "skip": -3.0,
    "dislike": -5.0
}

train["interaction_weight"] = (
    train["event"]
    .map(interaction_weights)
    .fillna(0)
)


# ============================================================
# 6. ADD COMPLETION INFORMATION
# ============================================================

# Completion rate gives additional information about preference.
#
# A song listened to for 95% is stronger evidence than
# a song listened to for only 20%.

train["completion_signal"] = (
    train["completion_rate"]
    .fillna(0)
)

train["interaction_weight"] = (
    train["interaction_weight"]
    + train["completion_signal"] * 2.0
)


# ============================================================
# 7. BUILD USER PROFILES
# ============================================================

user_profiles = {}

for user_id, user_history in train.groupby("user_id"):

    valid_history = user_history[
        user_history["track_id"].isin(
            song_feature_matrix.index
        )
    ]

    if len(valid_history) == 0:
        continue

    weighted_vectors = []
    weights = []

    for _, interaction in valid_history.iterrows():

        track_id = interaction["track_id"]
        weight = interaction["interaction_weight"]

        if track_id not in song_feature_matrix.index:
            continue

        song_vector = np.asarray(
            song_feature_matrix.loc[track_id],
            dtype=float
        ).reshape(-1)

        if len(song_vector) != len(feature_columns):
            continue

        weighted_vectors.append(song_vector)

        weights.append(weight)

    weighted_vectors = np.array(
        weighted_vectors
    )

    weights = np.array(
        weights
    )

    # --------------------------------------------------------
    # Positive and negative interactions are handled separately
    # --------------------------------------------------------

    positive_mask = weights > 0
    negative_mask = weights < 0

    # Positive preference vector
    if positive_mask.any():

        positive_weights = weights[
            positive_mask
        ]

        positive_vectors = weighted_vectors[
            positive_mask
        ]

        positive_profile = np.average(
            positive_vectors,
            axis=0,
            weights=positive_weights
        )

    else:

        positive_profile = np.mean(
            weighted_vectors,
            axis=0
        )

    # Negative preference vector
    if negative_mask.any():

        negative_weights = np.abs(
            weights[negative_mask]
        )

        negative_vectors = weighted_vectors[
            negative_mask
        ]

        negative_profile = np.average(
            negative_vectors,
            axis=0,
            weights=negative_weights
        )

    else:

        negative_profile = np.zeros(
            len(feature_columns)
        )

    # --------------------------------------------------------
    # Final user taste profile
    # --------------------------------------------------------

    profile = (
        positive_profile
        - 0.5 * negative_profile
    )

    user_profiles[user_id] = profile


print(
    f"Created profiles for "
    f"{len(user_profiles):,} users"
)


# ============================================================
# 8. V2 RECOMMENDER
# ============================================================

def recommend_v2(
    user_id,
    current_song_features=None,
    n_recommendations=10,
    content_weight=0.40,
    user_weight=0.60
):

    if user_id not in user_profiles:

        raise ValueError(
            f"No profile found for user: {user_id}"
        )

    # --------------------------------------------------------
    # User preference vector
    # --------------------------------------------------------

    user_vector = user_profiles[user_id].reshape(
        1, -1
    )

    # Similarity between user and every song
    user_similarity = cosine_similarity(
        user_vector,
        song_feature_matrix.values
    )[0]

    # --------------------------------------------------------
    # Current-song similarity
    # --------------------------------------------------------

    if current_song_features is not None:

        current_song = pd.DataFrame(
            [current_song_features]
        )[feature_columns]

        current_song_scaled = scaler.transform(
            current_song
        )

        content_similarity = cosine_similarity(
            current_song_scaled,
            song_feature_matrix.values
        )[0]

    else:

        # If there is no current song,
        # use only user preference.
        content_similarity = np.zeros(
            len(df)
        )

        content_weight = 0.0
        user_weight = 1.0

    # --------------------------------------------------------
    # Combine scores
    # --------------------------------------------------------

    final_score = (
        user_weight * user_similarity
        +
        content_weight * content_similarity
    )

    # --------------------------------------------------------
    # Don't recommend songs the user already interacted with
    # --------------------------------------------------------

    listened_tracks = set(
        train.loc[
            train["user_id"] == user_id,
            "track_id"
        ]
    )

    recommendation_scores = final_score.copy()

    for i, track_id in enumerate(
        df["track_id"]
    ):

        if track_id in listened_tracks:

            recommendation_scores[i] = -np.inf

    # --------------------------------------------------------
    # Get top recommendations
    # --------------------------------------------------------

    top_indices = np.argsort(
        recommendation_scores
    )[
        ::-1
    ][:n_recommendations]

    recommendations = df.iloc[
        top_indices
    ].copy()

    # --------------------------------------------------------
    # Add scores
    # --------------------------------------------------------

    recommendations["user_similarity"] = (
        user_similarity[top_indices]
    )

    recommendations["content_similarity"] = (
        content_similarity[top_indices]
    )

    recommendations["recommendation_score"] = (
        recommendation_scores[top_indices]
    )

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    result_columns = [
        "track_id",
        "track_name",
        "artist_name",
        "album_name",
        "year",
        "language",
        "popularity",
        "user_similarity",
        "content_similarity",
        "recommendation_score"
    ]

    result_columns = [
        col for col in result_columns
        if col in recommendations.columns
    ]

    return recommendations[
        result_columns
    ].reset_index(drop=True)


# ============================================================
# 9. TEST V2
# ============================================================

test_user = train["user_id"].iloc[0]

print(
    "Testing user:",
    test_user
)

# Example current song
current_song_features = {
    "acousticness": 0.18,
    "danceability": 0.76,
    "energy": 0.91,
    "instrumentalness": 0.00,
    "liveness": 0.10,
    "loudness": -4.2,
    "speechiness": 0.05,
    "tempo": 168.0,
    "valence": 0.82
}

recommendations = recommend_v2(
    user_id=test_user,
    current_song_features=current_song_features,
    n_recommendations=10
)

display(recommendations)


# ============================================================
# 10. SAVE V2 MODEL
# ============================================================

joblib.dump(
    scaler,
    "v2_scaler.pkl"
)

joblib.dump(
    user_profiles,
    "v2_user_profiles.pkl"
)

print("\nV2 model saved:")
print("  v2_scaler.pkl")
print("  v2_user_profiles.pkl")

Total songs: 29129
Unique track IDs: 29129
Duplicate track IDs: 0
Created profiles for 1,000 users
Testing user: U0001


,track_id,track_name,artist_name,album_name,year,language,popularity,user_similarity,content_similarity,recommendation_score
0,4bjqLzLKHmecWTtzIK8YhQ,Menaklukkan Dunia,Once Mekel Feat. Shakira Jasmine,Menaklukkan Dunia (Official Song Asian Games 2...,2018,English,7,0.632784,0.608581,0.623103
1,0cBJzQvIlPiUWfnygZPGx6,Menaklukkan Dunia,"Once Mekel, Shakira Jasmine",Menaklukkan Dunia,2018,English,37,0.632784,0.608581,0.623103
2,6yAGv2b4pF9NKRLzVLlK7M,Disco,Shakira Peach,Disco,2024,English,25,0.619601,0.576584,0.602394
3,5sQi44U78b4HJT4KgyT1Ub,Lo Imprescindible,Shakira,"Fijación Oral, Vol. 1",2005,English,39,0.744421,0.384126,0.600303
4,5eg1Qpctq17v9WQBkYsdJS,"Un Manam Thedum - From ""83 - Tamil""","Benny Dayal, Pritam",83 - Tamil (Original Motion Picture Soundtrack),2022,Hindi,8,0.589790,0.615675,0.600144
5,4g077yizjVdMB5HFl3rJRo,Good Tonight,"Daniel Pemberton, Anthony Ramos",The Bad Guys (Original Motion Picture Soundtrack),2022,English,47,0.650921,0.500041,0.590569
6,5XYNCCkYoFLegWZcQU9ecn,Good Tonight (from The Bad Guys),"Daniel Pemberton, Anthony Ramos",Good Tonight (from The Bad Guys),2022,English,49,0.654946,0.493582,0.590401
7,0r0epP5ducx4zHCcjOT8XN,Baby Tujhe Paap Lagega,"Himesh Reshammiya, Sachin-Jigar, Amitabh Bhatt...",Zara Hatke Zara Bachke (Original Motion Pictur...,2023,Hindi,11,0.520005,0.677796,0.583122
8,2G5rctJGEwclDFA0r8iNcZ,"Baby Tujhe Paap Lagega (From ""Zara Hatke Zara ...","Himesh Reshammiya, Sachin-Jigar, Amitabh Bhatt...","Baby Tujhe Paap Lagega (From ""Zara Hatke Zara ...",2023,Hindi,22,0.520005,0.677796,0.583122
9,2A17TySL9lcoRTA28w9yfB,La La La (Brazil 2014) (feat. Carlinhos Brown),"Shakira, Carlinhos Brown",The 2014 FIFA World Cup Official Album: One Lo...,2014,English,61,0.510180,0.691201,0.582589



V2 model saved:
  v2_scaler.pkl
  v2_user_profiles.pkl


In [11]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_pickle("song_catalog.pkl").copy()

train = pd.read_csv("interactions_train.csv")
test = pd.read_csv("interactions_test.csv")

feature_columns = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "tempo",
    "valence"
]

# Remove duplicate track IDs
df = (
    df
    .drop_duplicates(subset=["track_id"])
    .dropna(subset=feature_columns)
    .reset_index(drop=True)
)

# ============================================================
# CREATE SONG FEATURE MATRIX
# ============================================================

song_features = df[feature_columns].copy()

# Use the same scaler approach as V2
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

song_features_scaled = scaler.fit_transform(
    song_features
)

song_feature_matrix = pd.DataFrame(
    song_features_scaled,
    index=df["track_id"],
    columns=feature_columns
)

# ============================================================
# INTERACTION WEIGHTS
# ============================================================

interaction_weights = {
    "like": 5.0,
    "complete": 3.0,
    "play": 1.0,
    "skip": -3.0,
    "dislike": -5.0
}

train["interaction_weight"] = (
    train["event"]
    .map(interaction_weights)
    .fillna(0)
)

train["completion_signal"] = (
    train["completion_rate"]
    .fillna(0)
)

train["interaction_weight"] = (
    train["interaction_weight"]
    + train["completion_signal"] * 2.0
)

# ============================================================
# REBUILD USER PROFILES
# ============================================================

user_profiles = {}

for user_id, user_history in train.groupby("user_id"):

    valid_history = user_history[
        user_history["track_id"].isin(
            song_feature_matrix.index
        )
    ]

    if len(valid_history) == 0:
        continue

    vectors = []
    weights = []

    for _, interaction in valid_history.iterrows():

        track_id = interaction["track_id"]
        weight = interaction["interaction_weight"]

        vector = np.asarray(
            song_feature_matrix.loc[track_id],
            dtype=float
        ).reshape(-1)

        if len(vector) != len(feature_columns):
            continue

        vectors.append(vector)
        weights.append(weight)

    if len(vectors) == 0:
        continue

    vectors = np.asarray(vectors, dtype=float)
    weights = np.asarray(weights, dtype=float)

    positive_mask = weights > 0
    negative_mask = weights < 0

    # Positive profile
    if positive_mask.any():

        positive_profile = np.average(
            vectors[positive_mask],
            axis=0,
            weights=weights[positive_mask]
        )

    else:

        positive_profile = np.mean(
            vectors,
            axis=0
        )

    # Negative profile
    if negative_mask.any():

        negative_profile = np.average(
            vectors[negative_mask],
            axis=0,
            weights=np.abs(weights[negative_mask])
        )

    else:

        negative_profile = np.zeros(
            len(feature_columns)
        )

    # Final user profile
    user_profiles[user_id] = (
        positive_profile
        - 0.5 * negative_profile
    )

print(
    f"User profiles created: {len(user_profiles):,}"
)

# ============================================================
# PREPARE TEST POSITIVE INTERACTIONS
# ============================================================

positive_events = [
    "like",
    "complete"
]

test_positive = test[
    test["event"].isin(positive_events)
].copy()

print(
    f"Positive test interactions: "
    f"{len(test_positive):,}"
)

# ============================================================
# V2 EVALUATION
# ============================================================

K = 10

precision_scores = []
recall_scores = []
ndcg_scores = []
hit_rate_scores = []

evaluated_users = 0

# Cache matrix for faster calculation
all_song_matrix = song_feature_matrix.values
all_track_ids = df["track_id"].values

for user_id, user_test in test_positive.groupby("user_id"):

    if user_id not in user_profiles:
        continue

    # --------------------------------------------------------
    # Ground-truth songs
    # --------------------------------------------------------

    relevant_tracks = set(
        user_test["track_id"]
    )

    if len(relevant_tracks) == 0:
        continue

    # --------------------------------------------------------
    # User similarity
    # --------------------------------------------------------

    user_vector = user_profiles[user_id].reshape(
        1, -1
    )

    user_similarity = cosine_similarity(
        user_vector,
        all_song_matrix
    )[0]

    # --------------------------------------------------------
    # Content similarity
    #
    # For evaluation we don't know the "current song"
    # associated with the future test interaction.
    #
    # Therefore V2 evaluation here measures the
    # USER-PERSONALIZATION component.
    # --------------------------------------------------------

    final_scores = user_similarity.copy()

    # --------------------------------------------------------
    # Remove songs already present in training
    # --------------------------------------------------------

    seen_tracks = set(
        train.loc[
            train["user_id"] == user_id,
            "track_id"
        ]
    )

    for i, track_id in enumerate(all_track_ids):

        if track_id in seen_tracks:

            final_scores[i] = -np.inf

    # --------------------------------------------------------
    # Top K recommendations
    # --------------------------------------------------------

    top_indices = np.argsort(
        final_scores
    )[::-1][:K]

    recommended_tracks = set(
        all_track_ids[top_indices]
    )

    # --------------------------------------------------------
    # Precision@K
    # --------------------------------------------------------

    hits = len(
        recommended_tracks
        & relevant_tracks
    )

    precision = hits / K

    # --------------------------------------------------------
    # Recall@K
    # --------------------------------------------------------

    recall = hits / len(
        relevant_tracks
    )

    # --------------------------------------------------------
    # Hit Rate@K
    # --------------------------------------------------------

    hit_rate = 1 if hits > 0 else 0

    # --------------------------------------------------------
    # NDCG@K
    # --------------------------------------------------------

    dcg = 0.0

    for rank, track_id in enumerate(
        all_track_ids[top_indices],
        start=1
    ):

        if track_id in relevant_tracks:

            dcg += 1 / np.log2(
                rank + 1
            )

    # Ideal DCG
    ideal_hits = min(
        K,
        len(relevant_tracks)
    )

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(
            1,
            ideal_hits + 1
        )
    )

    ndcg = (
        dcg / idcg
        if idcg > 0
        else 0
    )

    # --------------------------------------------------------
    # Store metrics
    # --------------------------------------------------------

    precision_scores.append(
        precision
    )

    recall_scores.append(
        recall
    )

    ndcg_scores.append(
        ndcg
    )

    hit_rate_scores.append(
        hit_rate
    )

    evaluated_users += 1


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n" + "=" * 60)
print("V2 EVALUATION RESULTS")
print("=" * 60)

print(
    f"\nUsers evaluated: {evaluated_users:,}"
)

print(
    f"\nPrecision@{K}: "
    f"{np.mean(precision_scores):.4f}"
)

print(
    f"Recall@{K}: "
    f"{np.mean(recall_scores):.4f}"
)

print(
    f"NDCG@{K}: "
    f"{np.mean(ndcg_scores):.4f}"
)

print(
    f"Hit Rate@{K}: "
    f"{np.mean(hit_rate_scores):.4f}"
)

print("\n" + "=" * 60)

User profiles created: 1,000
Positive test interactions: 494

V2 EVALUATION RESULTS

Users evaluated: 348

Precision@10: 0.0000
Recall@10: 0.0000
NDCG@10: 0.0000
Hit Rate@10: 0.0000



In [13]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# 1. PUT YOUR ACTUAL FILE NAMES HERE
# ============================================================

EN_V1_MODEL_PATH = "knn_model_eng.pkl"
EN_V1_SCALER_PATH = "scaler_eng.pkl"
EN_CATALOG_PATH = "eng_song_catalog.pkl"

HI_V1_MODEL_PATH = "knn_model_hin.pkl"
HI_V1_SCALER_PATH = "scaler_hin.pkl"
HI_CATALOG_PATH = "hin_song_catalog.pkl"

V2_USER_PROFILES_PATH = "v2_user_profiles.pkl"


# ============================================================
# 2. FEATURES
# ============================================================

feature_columns = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "tempo",
    "valence"
]


# ============================================================
# 3. LOAD V1 MODELS
# ============================================================

en_v1_model = joblib.load(
    EN_V1_MODEL_PATH
)

en_v1_scaler = joblib.load(
    EN_V1_SCALER_PATH
)

en_catalog = pd.read_pickle(
    EN_CATALOG_PATH
)


hi_v1_model = joblib.load(
    HI_V1_MODEL_PATH
)

hi_v1_scaler = joblib.load(
    HI_V1_SCALER_PATH
)

hi_catalog = pd.read_pickle(
    HI_CATALOG_PATH
)


# ============================================================
# 4. LOAD V2 USER PROFILES
# ============================================================

user_profiles = joblib.load(
    V2_USER_PROFILES_PATH
)

print(
    f"V2 user profiles loaded: "
    f"{len(user_profiles):,}"
)


# ============================================================
# 5. LOAD TRAIN / TEST DATA
# ============================================================

train = pd.read_csv(
    "interactions_train.csv"
)

test = pd.read_csv(
    "interactions_test.csv"
)


# ============================================================
# 6. CLEAN CATALOGS
# ============================================================

en_catalog = (
    en_catalog
    .drop_duplicates(
        subset=["track_id"]
    )
    .dropna(
        subset=feature_columns
    )
    .reset_index(drop=True)
)

hi_catalog = (
    hi_catalog
    .drop_duplicates(
        subset=["track_id"]
    )
    .dropna(
        subset=feature_columns
    )
    .reset_index(drop=True)
)


# ============================================================
# 7. POSITIVE TEST INTERACTIONS
# ============================================================

positive_events = [
    "like",
    "complete"
]

test_positive = test[
    test["event"].isin(
        positive_events
    )
].copy()


print(
    f"Positive test interactions: "
    f"{len(test_positive):,}"
)


# ============================================================
# 8. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    recommendations,
    relevant_tracks,
    k=10
):

    recommendations = list(
        recommendations[:k]
    )

    relevant_tracks = set(
        relevant_tracks
    )

    if len(relevant_tracks) == 0:
        return 0, 0, 0, 0

    hits = [
        track
        for track in recommendations
        if track in relevant_tracks
    ]

    hit_count = len(hits)

    # Precision@K
    precision = (
        hit_count / k
    )

    # Recall@K
    recall = (
        hit_count /
        len(relevant_tracks)
    )

    # Hit Rate@K
    hit_rate = (
        1
        if hit_count > 0
        else 0
    )

    # NDCG@K
    dcg = 0.0

    for rank, track_id in enumerate(
        recommendations,
        start=1
    ):

        if track_id in relevant_tracks:

            dcg += (
                1 /
                np.log2(rank + 1)
            )

    ideal_hits = min(
        k,
        len(relevant_tracks)
    )

    idcg = sum(
        1 /
        np.log2(rank + 1)
        for rank in range(
            1,
            ideal_hits + 1
        )
    )

    ndcg = (
        dcg / idcg
        if idcg > 0
        else 0
    )

    return (
        precision,
        recall,
        ndcg,
        hit_rate
    )


# ============================================================
# 9. V1 CANDIDATE GENERATOR
# ============================================================

def get_v1_candidates(
    current_features,
    model,
    scaler,
    catalog,
    n=50
):

    X = catalog[
        feature_columns
    ]

    X_scaled = scaler.transform(
        X
    )

    current_scaled = scaler.transform(
        pd.DataFrame(
            [current_features],
            columns=feature_columns
        )
    )

    # --------------------------------------------------------
    # IMPORTANT:
    # This assumes your V1 model is a NearestNeighbors
    # model trained on the scaled feature matrix.
    # --------------------------------------------------------

    distances, indices = (
        model.kneighbors(
            current_scaled,
            n_neighbors=min(
                n + 1,
                len(catalog)
            )
        )
    )

    indices = indices[0]

    candidates = catalog.iloc[
        indices
    ].copy()

    # Remove exact current track if present
    current_track_id = (
        current_features.get(
            "track_id",
            None
        )
        if isinstance(
            current_features,
            dict
        )
        else None
    )

    if current_track_id is not None:

        candidates = candidates[
            candidates["track_id"]
            != current_track_id
        ]

    return candidates.head(n)


# ============================================================
# 10. V1 + V2 EVALUATION
# ============================================================

K = 10

v1_precision = []
v1_recall = []
v1_ndcg = []
v1_hit_rate = []

v2_precision = []
v2_recall = []
v2_ndcg = []
v2_hit_rate = []

evaluated_users = 0


# ============================================================
# 11. EVALUATE USER
# ============================================================

for user_id, user_test in test_positive.groupby(
    "user_id"
):

    # --------------------------------------------------------
    # Need a V2 profile
    # --------------------------------------------------------

    if user_id not in user_profiles:
        continue


    # --------------------------------------------------------
    # Select current song
    # --------------------------------------------------------

    current_row = user_test.iloc[0]

    current_track_id = (
        current_row["track_id"]
    )


    # --------------------------------------------------------
    # Find current song in either catalog
    # --------------------------------------------------------

    current_song = en_catalog[
        en_catalog["track_id"]
        == current_track_id
    ]

    language = "english"

    if len(current_song) == 0:

        current_song = hi_catalog[
            hi_catalog["track_id"]
            == current_track_id
        ]

        language = "hindi"


    if len(current_song) == 0:

        continue


    current_song = current_song.iloc[0]


    current_features = (
        current_song[
            feature_columns
        ].to_dict()
    )


    # ========================================================
    # V1 — 50 ENGLISH + 50 HINDI
    # ========================================================

    # --------------------------------------------------------
    # English candidates
    # --------------------------------------------------------

    en_X = en_catalog[
        feature_columns
    ]

    en_X_scaled = en_v1_scaler.transform(
        en_X
    )

    current_en_scaled = (
        en_v1_scaler.transform(
            pd.DataFrame(
                [current_features],
                columns=feature_columns
            )
        )
    )

    en_distances, en_indices = (
        en_v1_model.kneighbors(
            current_en_scaled,
            n_neighbors=min(
                51,
                len(en_catalog)
            )
        )
    )

    en_indices = en_indices[0]

    en_candidates = en_catalog.iloc[
        en_indices
    ].copy()

    en_candidates = en_candidates[
        en_candidates["track_id"]
        != current_track_id
    ].head(50)


    # --------------------------------------------------------
    # Hindi candidates
    # --------------------------------------------------------

    hi_X = hi_catalog[
        feature_columns
    ]

    hi_X_scaled = hi_v1_scaler.transform(
        hi_X
    )

    current_hi_scaled = (
        hi_v1_scaler.transform(
            pd.DataFrame(
                [current_features],
                columns=feature_columns
            )
        )
    )

    hi_distances, hi_indices = (
        hi_v1_model.kneighbors(
            current_hi_scaled,
            n_neighbors=min(
                51,
                len(hi_catalog)
            )
        )
    )

    hi_indices = hi_indices[0]

    hi_candidates = hi_catalog.iloc[
        hi_indices
    ].copy()

    hi_candidates = hi_candidates[
        hi_candidates["track_id"]
        != current_track_id
    ].head(50)


    # ========================================================
    # COMBINE 100 CANDIDATES
    # ========================================================

    candidates = pd.concat(
        [
            en_candidates,
            hi_candidates
        ],
        ignore_index=True
    )

    # Remove duplicate track IDs
    candidates = candidates.drop_duplicates(
        subset=["track_id"]
    ).reset_index(drop=True)


    if len(candidates) == 0:
        continue


    # ========================================================
    # REMOVE SONGS ALREADY SEEN IN TRAINING
    # ========================================================

    seen_tracks = set(
        train.loc[
            train["user_id"] == user_id,
            "track_id"
        ]
    )

    candidates = candidates[
        ~candidates["track_id"].isin(
            seen_tracks
        )
    ].reset_index(drop=True)


    if len(candidates) == 0:
        continue


    # ========================================================
    # V1 TOP 10
    #
    # Rank according to V1 distance.
    # Recalculate similarity on the combined candidates.
    # ========================================================

    candidate_matrix = (
        candidates[
            feature_columns
        ].values
    )

    current_vector = np.array(
        [
            current_features[f]
            for f in feature_columns
        ]
    ).reshape(1, -1)


    # Cosine similarity is only used here to preserve
    # the V1 ranking after combining English + Hindi.
    v1_candidate_similarity = (
        cosine_similarity(
            current_vector,
            candidate_matrix
        )[0]
    )

    v1_order = np.argsort(
        v1_candidate_similarity
    )[::-1]

    v1_recommendations = (
        candidates.iloc[
            v1_order[:K]
        ]["track_id"]
        .values
    )


    # ========================================================
    # V2 — PERSONALIZED RANKING
    # ========================================================

    user_vector = np.asarray(
        user_profiles[user_id],
        dtype=float
    ).reshape(1, -1)


    candidate_similarity = (
        cosine_similarity(
            current_vector,
            candidate_matrix
        )[0]
    )


    user_similarity = (
        cosine_similarity(
            user_vector,
            candidate_matrix
        )[0]
    )


    # --------------------------------------------------------
    # V2 score
    # --------------------------------------------------------

    USER_WEIGHT = 0.60
    CONTENT_WEIGHT = 0.40

    v2_scores = (
        USER_WEIGHT *
        user_similarity
        +
        CONTENT_WEIGHT *
        candidate_similarity
    )


    v2_order = np.argsort(
        v2_scores
    )[::-1]


    v2_recommendations = (
        candidates.iloc[
            v2_order[:K]
        ]["track_id"]
        .values
    )


    # ========================================================
    # GROUND TRUTH
    # ========================================================

    relevant_tracks = set(
        user_test["track_id"]
    )

    relevant_tracks.discard(
        current_track_id
    )

    if len(relevant_tracks) == 0:
        continue


    # ========================================================
    # V1 METRICS
    # ========================================================

    metrics = calculate_metrics(
        v1_recommendations,
        relevant_tracks,
        K
    )

    v1_precision.append(
        metrics[0]
    )

    v1_recall.append(
        metrics[1]
    )

    v1_ndcg.append(
        metrics[2]
    )

    v1_hit_rate.append(
        metrics[3]
    )


    # ========================================================
    # V2 METRICS
    # ========================================================

    metrics = calculate_metrics(
        v2_recommendations,
        relevant_tracks,
        K
    )

    v2_precision.append(
        metrics[0]
    )

    v2_recall.append(
        metrics[1]
    )

    v2_ndcg.append(
        metrics[2]
    )

    v2_hit_rate.append(
        metrics[3]
    )


    evaluated_users += 1


# ============================================================
# 12. RESULTS
# ============================================================

results = pd.DataFrame({

    "Metric": [
        "Precision@10",
        "Recall@10",
        "NDCG@10",
        "Hit Rate@10"
    ],

    "V1": [
        np.mean(v1_precision),
        np.mean(v1_recall),
        np.mean(v1_ndcg),
        np.mean(v1_hit_rate)
    ],

    "V2": [
        np.mean(v2_precision),
        np.mean(v2_recall),
        np.mean(v2_ndcg),
        np.mean(v2_hit_rate)
    ]
})


# ============================================================
# 13. V2 IMPROVEMENT
# ============================================================

results["V2 Improvement (%)"] = np.where(
    results["V1"] != 0,
    (
        (
            results["V2"]
            -
            results["V1"]
        )
        /
        results["V1"]
    ) * 100,
    np.nan
)


# ============================================================
# 14. DISPLAY
# ============================================================

print("\n" + "=" * 75)
print("V1 vs V2 EVALUATION")
print("=" * 75)

print(
    f"\nUsers evaluated: "
    f"{evaluated_users:,}"
)

print(
    "\nCandidate pool: "
    "50 English + 50 Hindi → V2 → Top 10"
)

display(
    results.style.format({
        "V1": "{:.4f}",
        "V2": "{:.4f}",
        "V2 Improvement (%)": "{:+.2f}%"
    })
)

print("=" * 75)

V2 user profiles loaded: 1,000
Positive test interactions: 494

V1 vs V2 EVALUATION

Users evaluated: 64

Candidate pool: 50 English + 50 Hindi → V2 → Top 10


,Metric,V1,V2,V2 Improvement (%)
0,Precision@10,0.0000,0.0000,+nan%
1,Recall@10,0.0000,0.0000,+nan%
2,NDCG@10,0.0000,0.0000,+nan%
3,Hit Rate@10,0.0000,0.0000,+nan%


In [14]:
import joblib

# ============================================================
# PACKAGE V2 MODEL
# ============================================================

v2_model = {
    "user_profiles": user_profiles,

    "scaler": scaler,

    "feature_columns": feature_columns,

    # V2 scoring weights
    "user_weight": 0.60,
    "content_weight": 0.40,

    # Negative preference strength
    "negative_weight": 0.50,

    # Model metadata
    "model_version": "V2",
    "model_type": "Personalized Content-Based Recommender"
}


# ============================================================
# SAVE
# ============================================================

joblib.dump(
    v2_model,
    "v2_model.pkl"
)


print("V2 model saved successfully!")
print("File: v2_model.pkl")
print(
    f"Users included: "
    f"{len(user_profiles):,}"
)
print(
    f"Features: "
    f"{len(feature_columns)}"
)

V2 model saved successfully!
File: v2_model.pkl
Users included: 1,000
Features: 9


In [16]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# 1. FILE PATHS — CHANGE THESE
# ============================================================

EN_V1_MODEL_PATH = "knn_model_eng.pkl"
EN_V1_SCALER_PATH = "scaler_eng.pkl"
EN_CATALOG_PATH = "eng_song_catalog.pkl"

HI_V1_MODEL_PATH = "knn_model_hin.pkl"
HI_V1_SCALER_PATH = "scaler_hin.pkl"
HI_CATALOG_PATH = "hin_song_catalog.pkl"

V2_MODEL_PATH = "v2_model.pkl"


# ============================================================
# 2. FEATURES
# ============================================================

feature_columns = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "tempo",
    "valence"
]


# ============================================================
# 3. LOAD V1
# ============================================================

en_v1_model = joblib.load(
    EN_V1_MODEL_PATH
)

en_v1_scaler = joblib.load(
    EN_V1_SCALER_PATH
)

en_catalog = pd.read_pickle(
    EN_CATALOG_PATH
)

hi_v1_model = joblib.load(
    HI_V1_MODEL_PATH
)

hi_v1_scaler = joblib.load(
    HI_V1_SCALER_PATH
)

hi_catalog = pd.read_pickle(
    HI_CATALOG_PATH
)


# ============================================================
# 4. LOAD V2
# ============================================================

v2_model = joblib.load(
    V2_MODEL_PATH
)

user_profiles = v2_model[
    "user_profiles"
]

v2_scaler = v2_model[
    "scaler"
]

v2_features = v2_model[
    "feature_columns"
]

USER_WEIGHT = v2_model[
    "user_weight"
]

CONTENT_WEIGHT = v2_model[
    "content_weight"
]


# ============================================================
# 5. CLEAN CATALOGS
# ============================================================

en_catalog = (
    en_catalog
    .drop_duplicates(
        subset=["track_id"]
    )
    .dropna(
        subset=feature_columns
    )
    .reset_index(drop=True)
)

hi_catalog = (
    hi_catalog
    .drop_duplicates(
        subset=["track_id"]
    )
    .dropna(
        subset=feature_columns
    )
    .reset_index(drop=True)
)


# ============================================================
# 6. ENTER USER ID
# ============================================================

user_id = input(
    "Enter user ID: "
).strip()


if user_id not in user_profiles:

    print(
        f"\nUser '{user_id}' does not have a V2 profile."
    )

    print(
        "\nAvailable example users:"
    )

    print(
        list(user_profiles.keys())[:10]
    )

    raise SystemExit


# ============================================================
# 7. ENTER CURRENT SONG FEATURES
# ============================================================

print("\nEnter current song features:\n")

current_features = {}

for feature in feature_columns:

    current_features[feature] = float(
        input(
            f"{feature}: "
        )
    )


# ============================================================
# 8. CREATE INPUT DATAFRAME
# ============================================================

current_df = pd.DataFrame(
    [current_features],
    columns=feature_columns
)


# ============================================================
# 9. V1 — ENGLISH 50 CANDIDATES
# ============================================================

en_input_scaled = (
    en_v1_scaler.transform(
        current_df
    )
)

en_distances, en_indices = (
    en_v1_model.kneighbors(
        en_input_scaled,
        n_neighbors=min(
            51,
            len(en_catalog)
        )
    )
)

en_indices = en_indices[0]

en_candidates = en_catalog.iloc[
    en_indices
].copy()

en_candidates = (
    en_candidates
    .drop_duplicates(
        subset=["track_id"]
    )
    .head(50)
)


# ============================================================
# 10. V1 — HINDI 50 CANDIDATES
# ============================================================

hi_input_scaled = (
    hi_v1_scaler.transform(
        current_df
    )
)

hi_distances, hi_indices = (
    hi_v1_model.kneighbors(
        hi_input_scaled,
        n_neighbors=min(
            51,
            len(hi_catalog)
        )
    )
)

hi_indices = hi_indices[0]

hi_candidates = hi_catalog.iloc[
    hi_indices
].copy()

hi_candidates = (
    hi_candidates
    .drop_duplicates(
        subset=["track_id"]
    )
    .head(50)
)


# ============================================================
# 11. COMBINE V1 CANDIDATES
# ============================================================

candidates = pd.concat(
    [
        en_candidates,
        hi_candidates
    ],
    ignore_index=True
)

candidates = (
    candidates
    .drop_duplicates(
        subset=["track_id"]
    )
    .reset_index(drop=True)
)


print(
    f"\nV1 generated "
    f"{len(candidates)} candidates."
)


# ============================================================
# 12. V1 CONTENT SIMILARITY
# ============================================================

candidate_matrix = (
    candidates[
        feature_columns
    ].values
)

current_vector = (
    current_df[
        feature_columns
    ].values
)


content_similarity = (
    cosine_similarity(
        current_vector,
        candidate_matrix
    )[0]
)


# ============================================================
# 13. V2 USER SIMILARITY
# ============================================================

user_vector = np.asarray(
    user_profiles[user_id],
    dtype=float
).reshape(1, -1)


candidate_vectors_scaled = (
    v2_scaler.transform(
        candidates[
            v2_features
        ]
    )
)


user_similarity = (
    cosine_similarity(
        user_vector,
        candidate_vectors_scaled
    )[0]
)


# ============================================================
# 14. V2 FINAL SCORE
# ============================================================

v2_scores = (
    USER_WEIGHT *
    user_similarity
    +
    CONTENT_WEIGHT *
    content_similarity
)


# ============================================================
# 15. RANK USING V2
# ============================================================

ranked_indices = np.argsort(
    v2_scores
)[::-1]


top_indices = ranked_indices[:10]


recommendations = (
    candidates.iloc[
        top_indices
    ].copy()
)


recommendations[
    "v2_score"
] = v2_scores[
    top_indices
]


# ============================================================
# 16. DISPLAY FINAL RESULT
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "V1 → V2 RECOMMENDATIONS"
)

print(
    "=" * 80
)

display_columns = [
    "track_id",
    "track_name",
    "artist_name",
    "album_name"
]

# Add only columns that actually exist
display_columns = [
    col
    for col in display_columns
    if col in recommendations.columns
]

display_columns.append(
    "v2_score"
)

print(
    f"\nUser: {user_id}"
)

print(
    f"V1 candidates: "
    f"{len(candidates)}"
)

print(
    "\nTop 10 recommendations:\n"
)

display(
    recommendations[
        display_columns
    ].reset_index(drop=True)
)

Enter user ID: U0001

Enter current song features:

acousticness: 0.636
danceability: 0.371
energy: 0.668
instrumentalness: 2e-06
liveness: 0.294
loudness: -7.269
speechiness: 0.0497
tempo: 87.458
valence: 0.403

V1 generated 100 candidates.

V1 → V2 RECOMMENDATIONS

User: U0001
V1 candidates: 100

Top 10 recommendations:



,track_id,track_name,artist_name,album_name,v2_score
0,03j354P848KtNU2FVSwkDG,Real Life,The Weeknd,Beauty Behind The Madness,0.887517
1,5GWxfgx9trlEuZK3VmTJHe,Pularkaalam Pole,"Haricharan, Madonna Sebastian",Magical Duets,0.846010
2,7rAE9dajt8p3xQKsVStSNw,Talk of Town,Dashboard Madonna,Talk Of Town,0.834684
3,0hxRewFYgFH2TEua7OX3jt,Maranthana - Lofi Beats,"Saindhavi, Justin Prabhakaran, Vivek, The Inde...",Maranthana (Lofi Beats),0.825571
4,1T12rhqTX3xerTbWa4NLdA,Arabu Naadu (LoFi Mix),"Haricharan, Yuvan Shankar Raja, DJ Aftab",Love Pro Max (Tamil),0.791265
5,5OPOFr8ciGJd8gUJ9CZTy9,Gotta Be You - Live,One Direction,Live - EP,0.774854
6,2u4NOGZ9DfNxERLwnLOrvW,"Summer, Highland Falls - Live at Shea Stadium,...",Billy Joel,Live At Shea Stadium,0.768961
7,0vv7WPmdbL3fkN8UsiiFqh,"Phir Kabhi (From ""M.S.Dhoni - The Untold Story"")",Arijit Singh,A Musical Tribute To Sushant Singh Rajput,0.764580
8,4jk4CaqBMBbMZhf3PuR1ai,Phir Kabhi,Arijit Singh,M.S.Dhoni - The Untold Story,0.763904
9,1nHKI4L5pWrN5CUvW07nHP,Let Her Go (feat. Ed Sheeran) - Anniversary Ed...,"Passenger, Ed Sheeran",All The Little Lights (Anniversary Edition),0.758535


In [ ]:
{'id': '15841412-7495-40f6-bb86-dee5cef22a58', 'href': 'https://open.spotify.com/track/7iLA6PQeQWRjLDHOJgGpj5', 'isrc': 'INS171501741', 'acousticness': 0.636, 'danceability': 0.371, 'energy': 0.668, 'instrumentalness': 2e-06, 'key': 11, 'liveness': 0.294, 'loudness': -7.269, 'mode': 0, 'speechiness': 0.0497, 'tempo': 87.458, 'valence': 0.403}